# Feature Transformation and Weight Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/concept_notebooks/Feature_Transformation_and_Weight_Learning.ipynb)

## Goal

This notebook teaches a core machine learning idea:

**Machines do not learn from raw columns blindly. They learn from numerical patterns, and those patterns become clearer after we study distributions and transform features appropriately.**

We will walk through a small synthetic student-performance dataset and answer four questions:

1. What does each feature distribution look like?
2. Which features need transformation?
3. Why do we standardize features before learning weights?
4. How do transformed features make model weights easier to interpret?


## Running in Colab or Kaggle

- This notebook is fully self-contained because it generates its own synthetic dataset.
- In **Google Colab**, open the notebook from GitHub or upload the `.ipynb` file directly.
- In **Kaggle**, create a new notebook and upload this file through the notebook interface.
- If any library is missing in a runtime, run the setup cell below once.


In [ ]:
# Optional setup for cloud notebook environments such as Colab or Kaggle
# Uncomment and run only if a package import fails.

# %pip install -q numpy pandas matplotlib seaborn scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

np.random.seed(42)
print('Libraries loaded.')

## Part 1 - Define the Learning Problem

Suppose we want to predict a student's **final exam score** from measurable inputs.

Our features will be:

- `study_hours_per_week`
- `attendance_percent`
- `practice_questions_solved`
- `sleep_hours`
- `previous_score`

Our target will be:

- `final_score`

The important teaching point is that these features do **not** all behave the same way:

- some are roughly symmetric
- some are bounded percentages
- some are right-skewed
- some live on very different numeric scales

That is exactly why we study distributions before training.

In [ ]:
# Create a small but realistic synthetic dataset
n = 220

study_hours = np.clip(np.random.normal(loc=12, scale=4, size=n), 1, 30)
attendance = np.clip(np.random.normal(loc=78, scale=12, size=n), 40, 100)
practice_questions = np.clip(np.random.exponential(scale=35, size=n), 0, 220)
sleep_hours = np.clip(np.random.normal(loc=6.8, scale=1.1, size=n), 4, 9)
previous_score = np.clip(np.random.normal(loc=64, scale=11, size=n), 30, 95)

noise = np.random.normal(loc=0, scale=5, size=n)
final_score = (
    18
    + 1.4 * study_hours
    + 0.22 * attendance
    + 3.2 * np.log1p(practice_questions)
    + 2.0 * sleep_hours
    + 0.45 * previous_score
    + noise
)
final_score = np.clip(final_score, 35, 100)

df = pd.DataFrame({
    'study_hours_per_week': study_hours,
    'attendance_percent': attendance,
    'practice_questions_solved': practice_questions,
    'sleep_hours': sleep_hours,
    'previous_score': previous_score,
    'final_score': final_score,
})

df.head()

In [ ]:
df.describe().T.round(2)

## Part 2 - Study Feature Distributions

A **distribution** tells us how values are spread out.

Before training a model, we ask:

- Where is the center of the feature?
- How wide is the spread?
- Is the distribution symmetric or skewed?
- Are there outliers?
- Is the feature on a much larger scale than the others?

These questions matter because machine learning algorithms learn numerical relationships. If the numeric representation is distorted, the learned weights can also become distorted.

In [ ]:
feature_cols = [
    'study_hours_per_week',
    'attendance_percent',
    'practice_questions_solved',
    'sleep_hours',
    'previous_score',
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#2a6f97')
    axes[i].set_title(col)

sns.histplot(df['final_score'], kde=True, ax=axes[-1], color='#ef476f')
axes[-1].set_title('final_score')

plt.tight_layout()
plt.show()

In [ ]:
distribution_summary = pd.DataFrame({
    'mean': df[feature_cols].mean(),
    'std_dev': df[feature_cols].std(),
    'min': df[feature_cols].min(),
    'max': df[feature_cols].max(),
    'skewness': df[feature_cols].skew(),
}).round(2)

distribution_summary

### What we notice

- `practice_questions_solved` is strongly right-skewed.
- `attendance_percent` is bounded between 40 and 100, so its shape is different from an unbounded variable.
- `study_hours_per_week` and `previous_score` are closer to symmetric.
- The scales are very different. One column ranges roughly from 0 to 200, another from 4 to 9.

This immediately suggests two preprocessing needs:

1. Transform skewed features so the model sees a more stable numerical pattern.
2. Standardize features so one column does not dominate simply because its numbers are larger.

## Part 3 - Transform the Features

### A. Log transformation for right-skewed data

When a feature has a long right tail, a few very large values can compress the rest of the data.

A common fix is:

$$x_{new} = \log(1 + x)$$

This keeps the order of the values but reduces the effect of very large observations.

### B. Standardization for weight learning

After shape issues are handled, we often standardize each feature:

$$z = \frac{x - \mu}{\sigma}$$

Now every feature is measured in the same unit: **standard deviations from its mean**.

This is powerful because a model weight on standardized features answers a clean question:

**If this feature increases by 1 standard deviation, how much does the prediction change?**

In [ ]:
df_transformed = df.copy()

# Transform the skewed feature first
df_transformed['practice_questions_log'] = np.log1p(df_transformed['practice_questions_solved'])

transformed_feature_cols = [
    'study_hours_per_week',
    'attendance_percent',
    'practice_questions_log',
    'sleep_hours',
    'previous_score',
]

# Z-score standardization
for col in transformed_feature_cols:
    mean = df_transformed[col].mean()
    std = df_transformed[col].std()
    df_transformed[col + '_z'] = (df_transformed[col] - mean) / std

df_transformed[[
    'practice_questions_solved',
    'practice_questions_log',
    'practice_questions_log_z'
]].head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df['practice_questions_solved'], kde=True, ax=axes[0], color='#ef476f')
axes[0].set_title('Raw practice_questions_solved')

sns.histplot(df_transformed['practice_questions_log'], kde=True, ax=axes[1], color='#118ab2')
axes[1].set_title('After log1p transform')

sns.histplot(df_transformed['practice_questions_log_z'], kde=True, ax=axes[2], color='#06d6a0')
axes[2].set_title('After log1p + z-score')

plt.tight_layout()
plt.show()

## Part 4 - Train Models Before and After Transformation

We will train two linear regression models:

1. A model on the raw features
2. A model on transformed and standardized features

Both models will try to predict `final_score`.

The accuracy may be similar, but the **meaning of the weights** will be much clearer after transformation.

In [ ]:
raw_X = df[feature_cols]
y = df['final_score']

z_cols = [col + '_z' for col in transformed_feature_cols]
transformed_X = df_transformed[z_cols]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(raw_X, y, test_size=0.25, random_state=7)
X_train_trans, X_test_trans, _, _ = train_test_split(transformed_X, y, test_size=0.25, random_state=7)

raw_model = LinearRegression()
raw_model.fit(X_train_raw, y_train)
raw_pred = raw_model.predict(X_test_raw)

trans_model = LinearRegression()
trans_model.fit(X_train_trans, y_train)
trans_pred = trans_model.predict(X_test_trans)

def regression_report(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return pd.Series({
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': rmse,
        'R2': r2_score(y_true, y_pred)
    }).round(3)

results = pd.DataFrame({
    'Raw feature model': regression_report(y_test, raw_pred),
    'Transformed feature model': regression_report(y_test, trans_pred)
})

results

In [ ]:
raw_weights = pd.Series(raw_model.coef_, index=feature_cols, name='raw_weight').round(3)
trans_weights = pd.Series(trans_model.coef_, index=transformed_feature_cols, name='standardized_weight').round(3)

pd.concat([raw_weights, trans_weights], axis=1)

## Part 5 - Interpreting the Learned Weights

### Raw-feature weights

A raw weight is hard to compare directly because each feature uses a different unit.

- `attendance_percent` changes in percentage points
- `sleep_hours` changes in hours
- `practice_questions_solved` changes in counts with strong skew

So a larger raw coefficient does **not automatically** mean a more important feature.

### Standardized-feature weights

After transformation and z-scoring, each coefficient is on a common scale.

Example interpretation:

- if the standardized weight for `previous_score` is 5.2,
- then increasing `previous_score` by 1 standard deviation increases the predicted `final_score` by about 5.2 points, holding the other features fixed.

This is why transformed features are useful for teaching and for model interpretation.

In [ ]:
new_student = pd.DataFrame({
    'study_hours_per_week': [14],
    'attendance_percent': [84],
    'practice_questions_solved': [42],
    'sleep_hours': [7.2],
    'previous_score': [68],
})

new_student_transformed = new_student.copy()
new_student_transformed['practice_questions_log'] = np.log1p(new_student_transformed['practice_questions_solved'])

for col in transformed_feature_cols:
    mean = df_transformed[col].mean()
    std = df_transformed[col].std()
    new_student_transformed[col + '_z'] = (new_student_transformed[col] - mean) / std

predicted_score = trans_model.predict(new_student_transformed[z_cols])[0]
print(f'Predicted final score: {predicted_score:.2f}')

## Key Takeaways

1. A dataset is not ready for learning just because it is in a table.
2. We inspect distributions to understand spread, skewness, outliers, and scale.
3. Skewed features often benefit from transformations such as `log1p`.
4. Standardization puts features onto a common statistical scale.
5. Once features are transformed, learned weights become easier to compare and explain.
6. This is how statistics supports machine learning: statistics prepares the representation, and learning algorithms estimate the weights.

### Suggested classroom discussion

- Which features were easiest for the model to learn from directly?
- Why is a skewed distribution a problem?
- Why are standardized coefficients easier to compare than raw coefficients?
- Would the same preprocessing choices always be correct for every model?
